# Architecture B-PR — Multi-agent Single-model with Prompt Repetition

This notebook runs Architecture **B** (multi-agent, single-model) with **Prompt Repetition** enabled.

**Prompt Repetition**: Based on Leviathan et al., "Prompt Repetition Improves Non-Reasoning LLMs" (arXiv:2512.14982, 2025).
The technique repeats user prompts (`<QUERY>` → `<QUERY>\n\n<QUERY>`) to allow each token to attend to all other tokens.

**Architecture B**: Planner → Router → Developer (S/M/L) → Reviewer → Tester, all using Qwen-7B.

In [1]:
import os
import sys
import subprocess
import pathlib

REPO_URL = "https://github.com/LLM4SE-group-15/ArchitecturesForCodeDevelopmentWithLLMs.git"
REPO_DIR = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print(f"Using repo at {REPO_DIR.resolve()}")

Cloning into '/content/ArchitecturesForCodeDevelopmentWithLLMs'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 12.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.4/536.4 k

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
fastai

In [2]:
!pip install -r requirements.txt

In [ ]:
import os
import getpass
from huggingface_hub import login

# Configurazione LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "project_name"
os.environ["LANGCHAIN_API_KEY"] = "langchain_api_key"

# Forza sempre l'uso del token che inserisci
os.environ["HF_TOKEN"] = "hf_token"
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

os.environ["ARCHITECTURE"] = "B"
os.environ["PROMPT_REPETITION"] = "true"  # Enable prompt repetition for RQ4
print("ARCHITECTURE set to", os.environ["ARCHITECTURE"])
print("PROMPT_REPETITION enabled")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


ARCHITECTURE set to B
PROMPT_REPETITION enabled


In [4]:
from huggingface_hub import HfApi

api = HfApi()
try:
    user_info = api.whoami(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face as:", user_info.get("name") or user_info.get("user"))
except Exception as exc:
    print("Login check failed:", exc)

Logged in to Hugging Face as: Riaburger


In [5]:
import json
import time
import logging
import pathlib
import os

from datetime import datetime

DEFAULT_ROOT = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")
ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else pathlib.Path.cwd()

sys.path.insert(0, str(ROOT))

LOG_DIR = ROOT / "log"
LOG_DIR.mkdir(exist_ok=True)

logger = logging.getLogger("architecture_B_PR")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "architecture_B_PR.log")
stream_handler = logging.StreamHandler()
for handler in (file_handler, stream_handler):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger ready. Repo root: %s", ROOT)
logger.info("Log files: %s", LOG_DIR)
logger.info("PROMPT_REPETITION: %s", os.environ.get("PROMPT_REPETITION", "false"))
print("Logs ->", LOG_DIR)

2026-01-28 13:18:51,396 | INFO | Logger ready. Repo root: /content/ArchitecturesForCodeDevelopmentWithLLMs
2026-01-28 13:18:51,396 | INFO | Log files: /content/ArchitecturesForCodeDevelopmentWithLLMs/log
2026-01-28 13:18:51,397 | INFO | PROMPT_REPETITION: true


Logs -> /content/ArchitecturesForCodeDevelopmentWithLLMs/log


In [6]:
import time
import json
import random
from src.data.task_loader import HumanEvalTaskLoader
from src.graph.graph import run_graph
from src.agents.llm import Architecture

ARCH = Architecture.B
SEED_FIXED = 31

def run_humaneval_benchmark(limit: int = 15, shuffle: bool = True):
    """
    Run benchmark on HumanEval tasks with Prompt Repetition enabled.
    
    Args:
        limit: Number of tasks to run
        shuffle: If True, randomly sample tasks with fixed seed
    """
    loader = HumanEvalTaskLoader()
    all_tasks = loader.load_all()
    
    if shuffle:
        random.seed(SEED_FIXED)
        tasks = random.sample(all_tasks, min(limit, len(all_tasks)))
    else:
        tasks = all_tasks[:limit]

    results = []
    total = len(tasks)
    logger.info("Loaded %s tasks from HumanEval (shuffle=%s, seed=%s)", total, shuffle, SEED_FIXED)
    logger.info("Prompt Repetition: ENABLED")
    print(f"Starting benchmark on {total} tasks (Prompt Repetition: ON)...")
    
    for idx, task in enumerate(tasks, 1):
        logger.info("Running %s/%s %s", idx, total, task.task_id)
        print(f"[{idx}/{total}] Task {task.task_id} ({task.entry_point})... ", end="", flush=True)
        
        start = time.time()
        
        state = run_graph(
            task_id=task.task_id,
            task_description=task.prompt,
            test_code=task.test,
            entry_point=task.entry_point,
            architecture=ARCH,
        )

        elapsed = time.time() - start
        
        record = {
            "task_id": task.task_id,
            "entry_point": task.entry_point,
            "architecture": "B-PR",
            "prompt_repetition": True,
            "test_passed": state["test_passed"],
            "developer_tier": state.get("developer_tier"),
            "escalations": state["escalations"],
            "story_points_initial": state.get("story_points_initial"),
            "story_points_final": state.get("story_points_current"),
            "elapsed_seconds": elapsed,
            "generated_code": state.get("generated_code", ""),
        }
        results.append(record)
        
        logger.info(
            "Finished %s | pass=%s tier=%s escalations=%s elapsed=%.1fs",
            task.task_id,
            state["test_passed"],
            record["developer_tier"],
            record["escalations"],
            elapsed,
        )
        status_str = "PASS" if state["test_passed"] else "FAIL"
        print(f"{status_str} in {elapsed:.1f}s")

        with open(LOG_DIR / "architecture_B_PR.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
            
    return results

# Esecuzione: 15 task random (seed=31 per riproducibilita)
sample_results = run_humaneval_benchmark(limit=164, shuffle=False)

# Sommario
passed_count = sum(1 for r in sample_results if r['test_passed'])
print(f"\nBenchmark Completed. Passed: {passed_count}/{len(sample_results)}")

Loading HumanEval dataset...


Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

2026-01-28 13:18:56,623 | INFO | Loaded 164 tasks from HumanEval (shuffle=False, seed=31)
2026-01-28 13:18:56,624 | INFO | Prompt Repetition: ENABLED
2026-01-28 13:18:56,624 | INFO | Running 1/164 HumanEval/0


Loaded 164 tasks.
Starting benchmark on 164 tasks (Prompt Repetition: ON)...
[1/164] Task HumanEval/0 (has_close_elements)... 

2026-01-28 13:19:09,733 | INFO | Finished HumanEval/0 | pass=False tier=L escalations=1 elapsed=13.1s
2026-01-28 13:19:09,734 | INFO | Running 2/164 HumanEval/1


...

PASS in 12.0s
[163/164] Task HumanEval/162 (string_to_md5)... 

2026-01-28 13:51:58,351 | INFO | Finished HumanEval/162 | pass=True tier=S escalations=0 elapsed=6.7s
2026-01-28 13:51:58,352 | INFO | Running 164/164 HumanEval/163


PASS in 6.7s
[164/164] Task HumanEval/163 (generate_integers)... 

2026-01-28 13:52:23,348 | INFO | Finished HumanEval/163 | pass=False tier=L escalations=2 elapsed=25.0s


FAIL in 25.0s

Benchmark Completed. Passed: 109/164


In [7]:
!cd log && cat architecture_B_PR.jsonl

{"task_id": "HumanEval/0", "entry_point": "has_close_elements", "architecture": "B-PR", "prompt_repetition": true, "test_passed": false, "developer_tier": "L", "escalations": 1, "story_points_initial": 3, "story_points_final": 8, "elapsed_seconds": 13.108015298843384, "generated_code": "from typing import List\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:"}
{"task_id": "HumanEval/1", "entry_point": "separate_paren_groups", "architecture": "B-PR", "prompt_repetition": true, "test_passed": false, "developer_tier": "L", "escalations": 1, "story_points_initial": 3, "story_points_final": 8, "elapsed_seconds": 13.724416971206665, "generated_code": "from typing import List\n\n\ndef separate_paren_groups(paren_string: str) -> List[str]:"}
{"task_id": "HumanEval/2", "entry_point": "truncate_number", "architecture": "B-PR", "prompt_repetition": true, "test_passed": true, "developer_tier": "S", "escalations": 0, "story_points_initial": 1, "story_points_final": 1, "ela

## Evaluation Metrics for Architecture B-PR

This section calculates the evaluation metrics as specified in `evaluation.md`:

- **Primary Metrics**: Pass Rate, Pass@1
- **Cost Metrics**: Execution Time, API Calls, Escalations
- **Comparison**: B vs B-PR (RQ4 - Prompt Repetition effect)

In [8]:
import json
import pandas as pd

# Load results
results_file = LOG_DIR / "architecture_B_PR.jsonl"
records = []
with open(results_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} task results")
df

Loaded 164 task results


,task_id,entry_point,architecture,prompt_repetition,test_passed,developer_tier,escalations,story_points_initial,story_points_final,elapsed_seconds,generated_code
0,HumanEval/0,has_close_elements,B-PR,True,False,L,1,3,8,13.108015,from typing import List\n\ndef has_close_eleme...
1,HumanEval/1,separate_paren_groups,B-PR,True,False,L,1,3,8,13.724417,from typing import List\n\n\ndef separate_pare...
2,HumanEval/2,truncate_number,B-PR,True,True,S,0,1,1,5.950149,def truncate_number(number: float) -> float:\n...
3,HumanEval/3,below_zero,B-PR,True,False,L,2,2,8,17.192734,from typing import List\n\n\ndef below_zero(op...
4,HumanEval/4,mean_absolute_deviation,B-PR,True,True,L,2,2,8,15.761647,from typing import List\n\ndef mean_absolute_d...
...,...,...,...,...,...,...,...,...,...,...,...
159,HumanEval/159,eat,B-PR,True,True,S,0,2,2,7.815571,"def eat(number, need, remaining):\n """"""\n ..."
160,HumanEval/160,do_algebra,B-PR,True,True,M,0,3,3,6.967377,import operator\n\ndef do_algebra(operator_lis...
161,HumanEval/161,solve,B-PR,True,True,L,1,3,8,12.008660,def solve(s):\n has_letters = any(c.isalpha...
162,HumanEval/162,string_to_md5,B-PR,True,True,S,0,1,1,6.744511,import hashlib\n\ndef string_to_md5(text):\n ...


In [9]:
# Static Code Quality Metrics (Radon)
# Calculates Cyclomatic Complexity and Maintainability Index for generated code

from radon.complexity import cc_visit
from radon.metrics import mi_visit

def calculate_static_metrics(code: str) -> dict:
    """Calculate static code quality metrics using Radon."""
    if not code or not code.strip():
        return {
            "cyclomatic_complexity_avg": None, 
            "cyclomatic_complexity_max": None,
            "maintainability_index": None
        }
    
    try:
        # Cyclomatic Complexity - average across all functions
        cc_results = cc_visit(code)
        if cc_results:
            avg_cc = sum(block.complexity for block in cc_results) / len(cc_results)
            max_cc = max(block.complexity for block in cc_results)
        else:
            avg_cc = 1  # No functions = simple code
            max_cc = 1
    except Exception:
        avg_cc = None
        max_cc = None
    
    try:
        # Maintainability Index (0-100, higher is better)
        mi_score = mi_visit(code, multi=False)
    except Exception:
        mi_score = None
    
    return {
        "cyclomatic_complexity_avg": avg_cc,
        "cyclomatic_complexity_max": max_cc,
        "maintainability_index": mi_score
    }

# Calculate metrics for all generated code
print("\nCalculating static code quality metrics...")
static_metrics = []
for idx, row in df.iterrows():
    code = row.get("generated_code", "")
    metrics = calculate_static_metrics(code)
    metrics["task_id"] = row["task_id"]
    metrics["test_passed"] = row["test_passed"]
    static_metrics.append(metrics)

metrics_df = pd.DataFrame(static_metrics)

# Add to main dataframe
df["cyclomatic_complexity_avg"] = metrics_df["cyclomatic_complexity_avg"]
df["cyclomatic_complexity_max"] = metrics_df["cyclomatic_complexity_max"]
df["maintainability_index"] = metrics_df["maintainability_index"]

# Summary statistics
valid_cc = metrics_df["cyclomatic_complexity_avg"].dropna()
valid_mi = metrics_df["maintainability_index"].dropna()

print("\n" + "=" * 60)
print("STATIC CODE QUALITY METRICS")
print("=" * 60)
print(f"\nCyclomatic Complexity (lower is better):")
print(f"  Average CC: {valid_cc.mean():.2f}" if len(valid_cc) > 0 else "  Average CC: N/A")
print(f"  Median CC: {valid_cc.median():.2f}" if len(valid_cc) > 0 else "  Median CC: N/A")
print(f"  Max CC: {metrics_df['cyclomatic_complexity_max'].max():.2f}" if metrics_df['cyclomatic_complexity_max'].notna().any() else "  Max CC: N/A")
print(f"\nMaintainability Index (0-100, higher is better):")
print(f"  Average MI: {valid_mi.mean():.2f}" if len(valid_mi) > 0 else "  Average MI: N/A")
print(f"  Median MI: {valid_mi.median():.2f}" if len(valid_mi) > 0 else "  Median MI: N/A")
print(f"  Min MI: {valid_mi.min():.2f}" if len(valid_mi) > 0 else "  Min MI: N/A")
print("=" * 60)

# Passed vs Failed comparison
passed_df = metrics_df[metrics_df["test_passed"] == True]
failed_df = metrics_df[metrics_df["test_passed"] == False]

print(f"\nComparison - Passed vs Failed Tasks:")
print(f"  Passed tasks - Avg CC: {passed_df['cyclomatic_complexity_avg'].mean():.2f}, Avg MI: {passed_df['maintainability_index'].mean():.2f}" if len(passed_df) > 0 else "  Passed tasks: No data")
print(f"  Failed tasks - Avg CC: {failed_df['cyclomatic_complexity_avg'].mean():.2f}, Avg MI: {failed_df['maintainability_index'].mean():.2f}" if len(failed_df) > 0 else "  Failed tasks: No data")



Calculating static code quality metrics...

STATIC CODE QUALITY METRICS

Cyclomatic Complexity (lower is better):
  Average CC: 4.14
  Median CC: 4.00
  Max CC: 14.00

Maintainability Index (0-100, higher is better):
  Average MI: 84.69
  Median MI: 88.85
  Min MI: 50.85

Comparison - Passed vs Failed Tasks:
  Passed tasks - Avg CC: 3.90, Avg MI: 85.24
  Failed tasks - Avg CC: 5.58, Avg MI: 81.31


In [10]:
# Calculate metrics
total_tasks = len(df)
passed_tasks = df['test_passed'].sum()
pass_rate = passed_tasks / total_tasks * 100
avg_time = df['elapsed_seconds'].mean()
total_time = df['elapsed_seconds'].sum()
avg_escalations = df['escalations'].mean()

print("=" * 50)
print("ARCHITECTURE B-PR (Multi-agent + Prompt Repetition)")
print("=" * 50)
print(f"Total Tasks:     {total_tasks}")
print(f"Passed:          {passed_tasks}")
print(f"Pass Rate:       {pass_rate:.1f}%")
print(f"Avg Time/Task:   {avg_time:.2f}s")
print(f"Total Time:      {total_time:.1f}s")
print(f"Avg Escalations: {avg_escalations:.2f}")
print("=" * 50)
print("\nPrompt Repetition: ENABLED")

ARCHITECTURE B-PR (Multi-agent + Prompt Repetition)
Total Tasks:     164
Passed:          109
Pass Rate:       66.5%
Avg Time/Task:   12.23s
Total Time:      2006.3s
Avg Escalations: 0.85

Prompt Repetition: ENABLED


In [11]:
# Tier distribution
print("\nDeveloper Tier Distribution:")
print(df['developer_tier'].value_counts())


Developer Tier Distribution:
developer_tier
L    71
S    49
M    44
Name: count, dtype: int64


In [12]:
# Verifica prompt repetition
from src.agents.client import get_llm_client
from src.agents.llm import get_prompt_repetition

print(f"PROMPT_REPETITION env: {os.environ.get('PROMPT_REPETITION')}")
print(f"get_prompt_repetition(): {get_prompt_repetition()}")

client = get_llm_client()
print(f"client.prompt_repetition: {client.prompt_repetition}")

# Test ripetizione
test_messages = [{"role": "user", "content": "Hello world"}]
repeated = client._apply_prompt_repetition(test_messages)
print(f"\nOriginal: {test_messages[0]['content']}")
print(f"Repeated: {repeated[0]['content']}")

PROMPT_REPETITION env: true
get_prompt_repetition(): True
client.prompt_repetition: True

Original: Hello world
Repeated: Hello world

Hello world
